# 02 — Data Preprocessing

This notebook prepares the Recife meteorological dataset for future Machine Learning experiments.

The preprocessing pipeline:

- loads the dataset;
- standardizes column names;
- converts meteorological variables to numeric types;
- creates a UTC datetime index;
- sorts observations chronologically;
- removes duplicated timestamps;
- reconstructs the complete hourly timeline;
- explicitly identifies missing observations;
- saves a reproducible preprocessed dataset.

Missing meteorological observations are not imputed at this stage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. FILE PATHS
# ============================================================

INPUT_PATH = Path("../data/processed/DataFrameRecife.csv")
OUTPUT_PATH = Path("../data/processed/recife_2018_preprocessed.csv")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {INPUT_PATH}")


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv(
    INPUT_PATH,
    sep=",",
    encoding="utf-8"
)

print("Original dataset shape:", df.shape)


# ============================================================
# 3. CLEAN ORIGINAL COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.replace("\n", "", regex=False)
)


# ============================================================
# 4. STANDARDIZE COLUMN NAMES
# ============================================================

column_mapping = {
    "Data": "date",
    "HORA (UTC)": "time_utc",
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)": "precipitation_mm",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)": "pressure_station_mb",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)": "pressure_max_prev_hour_mb",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)": "pressure_min_prev_hour_mb",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)": "temperature_c",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)": "dew_point_c",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)": "temperature_max_prev_hour_c",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)": "temperature_min_prev_hour_c",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)": "dew_point_max_prev_hour_c",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)": "dew_point_min_prev_hour_c",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)": "humidity_max_prev_hour_pct",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)": "humidity_min_prev_hour_pct",
    "UMIDADE RELATIVA DO AR, HORARIA (%)": "humidity_pct",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))": "wind_direction_deg",
    "VENTO, RAJADA MAXIMA (m/s)": "wind_gust_ms",
    "VENTO, VELOCIDADE HORARIA (m/s)": "wind_speed_ms",
}

df = df.rename(columns=column_mapping)


# ============================================================
# 5. VALIDATE EXPECTED COLUMNS
# ============================================================

expected_columns = [
    "date",
    "time_utc",
    "precipitation_mm",
    "pressure_station_mb",
    "pressure_max_prev_hour_mb",
    "pressure_min_prev_hour_mb",
    "temperature_c",
    "dew_point_c",
    "temperature_max_prev_hour_c",
    "temperature_min_prev_hour_c",
    "dew_point_max_prev_hour_c",
    "dew_point_min_prev_hour_c",
    "humidity_max_prev_hour_pct",
    "humidity_min_prev_hour_pct",
    "humidity_pct",
    "wind_direction_deg",
    "wind_gust_ms",
    "wind_speed_ms",
]

missing_columns = [
    column
    for column in expected_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Expected columns not found: {missing_columns}"
    )


# ============================================================
# 6. CONVERT METEOROLOGICAL VARIABLES TO NUMERIC
# ============================================================

numeric_columns = [
    column
    for column in expected_columns
    if column not in ["date", "time_utc"]
]

for column in numeric_columns:
    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 7. CREATE UTC DATETIME
# ============================================================

df["datetime_utc"] = pd.to_datetime(
    df["date"].astype(str)
    + " "
    + df["time_utc"].astype(str),
    format="%Y-%m-%d %H:%M",
    errors="coerce",
    utc=True
)

invalid_datetime_count = df["datetime_utc"].isna().sum()

if invalid_datetime_count > 0:
    raise ValueError(
        f"{invalid_datetime_count} timestamps could not be parsed."
    )


# ============================================================
# 8. SORT AND REMOVE DUPLICATED TIMESTAMPS
# ============================================================

df = (
    df
    .sort_values("datetime_utc")
    .drop_duplicates(
        subset="datetime_utc",
        keep="first"
    )
    .reset_index(drop=True)
)


# ============================================================
# 9. CREATE COMPLETE HOURLY TIMELINE FOR 2018
# ============================================================

full_hourly_index = pd.date_range(
    start="2018-01-01 00:00",
    end="2018-12-31 23:00",
    freq="h",
    tz="UTC"
)

df = df.set_index("datetime_utc")

df_preprocessed = df.reindex(full_hourly_index)

df_preprocessed.index.name = "datetime_utc"


# ============================================================
# 10. IDENTIFY OBSERVED AND MISSING HOURS
# ============================================================

df_preprocessed["is_observed"] = (
    df_preprocessed["date"].notna()
)

df_preprocessed["is_missing_hour"] = (
    ~df_preprocessed["is_observed"]
)


# ============================================================
# 11. RECONSTRUCT DATE AND TIME FROM DATETIME INDEX
# ============================================================

df_preprocessed["date"] = (
    df_preprocessed.index.strftime("%Y-%m-%d")
)

df_preprocessed["time_utc"] = (
    df_preprocessed.index.strftime("%H:%M")
)


# ============================================================
# 12. REORDER COLUMNS
# ============================================================

final_columns = [
    "date",
    "time_utc",
    "is_observed",
    "is_missing_hour",
    "precipitation_mm",
    "pressure_station_mb",
    "pressure_max_prev_hour_mb",
    "pressure_min_prev_hour_mb",
    "temperature_c",
    "dew_point_c",
    "temperature_max_prev_hour_c",
    "temperature_min_prev_hour_c",
    "dew_point_max_prev_hour_c",
    "dew_point_min_prev_hour_c",
    "humidity_max_prev_hour_pct",
    "humidity_min_prev_hour_pct",
    "humidity_pct",
    "wind_direction_deg",
    "wind_gust_ms",
    "wind_speed_ms",
]

df_preprocessed = df_preprocessed[final_columns]


# ============================================================
# 13. SAVE PREPROCESSED DATASET
# ============================================================

df_preprocessed.to_csv(
    OUTPUT_PATH,
    index=True
)


# ============================================================
# 14. PREPROCESSING SUMMARY
# ============================================================

observed_hours = int(
    df_preprocessed["is_observed"].sum()
)

missing_hours = int(
    df_preprocessed["is_missing_hour"].sum()
)

print()
print("=== PREPROCESSING SUMMARY ===")
print(f"Original observations: {len(df)}")
print(f"Complete hourly timeline: {len(df_preprocessed)}")
print(f"Observed hours: {observed_hours}")
print(f"Missing hours: {missing_hours}")
print(
    f"Missing percentage: "
    f"{missing_hours / len(df_preprocessed) * 100:.2f}%"
)
print()
print(f"Output saved to: {OUTPUT_PATH}")

display(df_preprocessed.head())

Original dataset shape: (8279, 18)

=== PREPROCESSING SUMMARY ===
Original observations: 8279
Complete hourly timeline: 8760
Observed hours: 8279
Missing hours: 481
Missing percentage: 5.49%

Output saved to: ..\data\processed\recife_2018_preprocessed.csv


,date,time_utc,is_observed,is_missing_hour,precipitation_mm,pressure_station_mb,pressure_max_prev_hour_mb,pressure_min_prev_hour_mb,temperature_c,dew_point_c,temperature_max_prev_hour_c,temperature_min_prev_hour_c,dew_point_max_prev_hour_c,dew_point_min_prev_hour_c,humidity_max_prev_hour_pct,humidity_min_prev_hour_pct,humidity_pct,wind_direction_deg,wind_gust_ms,wind_speed_ms
datetime_utc,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00+00:00,2018-01-01,00:00,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 01:00:00+00:00,2018-01-01,01:00,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 02:00:00+00:00,2018-01-01,02:00,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 03:00:00+00:00,2018-01-01,03:00,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 04:00:00+00:00,2018-01-01,04:00,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Conclusion

The preprocessing stage reconstructed the complete hourly timeline for 2018 and explicitly identified missing observations instead of treating absent timestamps as zero rainfall.

The dataset now contains:

- a standardized column structure;
- numeric meteorological variables;
- a UTC datetime index;
- duplicated timestamps removed;
- a complete hourly timeline;
- explicit indicators for observed and missing hours.

No meteorological values were imputed during this stage.

The resulting preprocessed dataset is ready for the next stage of the project: feature engineering and construction of future rainfall prediction targets.